# 01 — Exploratory Data Analysis (EDA)

**Purpose:** before any deeper analysis, validate that the two source tables are
fit for the **TOBi routing & escalation** business question. Per data-science
best practice, this notebook explores the data, surfaces quality issues, and
documents the assumptions on which the rest of the work depends.

**Scope (the two source tables):**
- `vf-pt-copsvertex-live.vfpt_dh_lake_cops_pub_investigation.f_kafka_tobi_sessions`  — one row per session
- `vf-pt-copsvertex-live.vfpt_dh_lake_cops_pub_investigation.f_tobi_logs_vertex` — one row per log token within a session

**What the reviewer should expect to find here:**
1. Volumes & date coverage of both tables.
2. Distributions of the key categorical fields (channels, service type, intents, outcome).
3. Null/quality issues that affect downstream classification.
4. Join integrity between sessions and logs.
5. The `LOG` token vocabulary used to reconstruct conversation flow.
6. EDA findings + decisions taken before the analysis.

## 1 — Setup & connect

In [ ]:
# %pip install google-cloud-bigquery db-dtypes pandas matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJECT = 'vf-pt-copsvertex-live'
SOURCE  = f'{PROJECT}.vfpt_dh_lake_cops_pub_investigation'
SES = f'`{SOURCE}.f_kafka_tobi_sessions`'
LOG = f'`{SOURCE}.f_tobi_logs_vertex`'

client = bigquery.Client(project=PROJECT)
LOCATION = client.get_dataset(SOURCE).location
print('source region =', LOCATION)
def q(sql): return client.query(sql, location=LOCATION).to_dataframe()

pd.set_option('display.max_columns', 60)
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True, 'grid.alpha':0.3})

## 2 — Sessions table: volume & coverage

**Why:** confirm the dataset is the size we expect for the time window and that
the timestamps are sane before computing anything else.

In [ ]:
q(f'''
SELECT COUNT(*)                          AS total_rows,
       COUNT(DISTINCT SESSION_ID)        AS distinct_sessions,
       MIN(START_MOMENT)                AS earliest_session,
       MAX(END_MOMENT)                  AS latest_session,
       COUNTIF(NEXT_SESSION_ID IS NOT NULL AND NEXT_SESSION_ID != '')      AS sessions_with_next,
       COUNTIF(INTERNAL_SES_LIST IS NOT NULL AND INTERNAL_SES_LIST != '') AS sessions_with_internal
FROM {SES}''')

## 3 — Logs table: volume & coverage

**Why:** the logs reconstruct the conversation flow. We need to know the volume,
the number of distinct sessions covered, and the average tokens per session.

In [ ]:
q(f'''
SELECT COUNT(*)                                          AS total_log_rows,
       COUNT(DISTINCT SESSION_ID)                       AS distinct_log_sessions,
       MIN(MOMENT)                                      AS earliest_log,
       MAX(MOMENT)                                      AS latest_log,
       ROUND(COUNT(*) / COUNT(DISTINCT SESSION_ID), 1)   AS avg_tokens_per_session
FROM {LOG}''')

## 4 — Join integrity (sessions <-> logs)

**Why:** the analysis joins the two tables on `SESSION_ID`. We need to confirm
the join is clean: log sessions exist in sessions, and most sessions have logs.

In [ ]:
q(f'''
WITH log_ids  AS (SELECT DISTINCT SESSION_ID FROM {LOG}),
     sess_ids AS (SELECT DISTINCT SESSION_ID FROM {SES})
SELECT (SELECT COUNT(*) FROM log_ids l JOIN sess_ids s USING(SESSION_ID))   AS log_sessions_matched,
       (SELECT COUNT(*) FROM log_ids l LEFT JOIN sess_ids s USING(SESSION_ID) WHERE s.SESSION_ID IS NULL) AS log_sessions_unmatched,
       (SELECT COUNT(*) FROM sess_ids s LEFT JOIN log_ids l USING(SESSION_ID) WHERE l.SESSION_ID IS NULL) AS sessions_without_logs
''')

## 5 — Date quality check

**Why:** during analysis we noticed `null` and `1900-01-01` dates in the raw
data. This step quantifies them so we can apply a clean analysis window.

In [ ]:
q(f'''
SELECT CASE WHEN START_MOMENT IS NULL                                 THEN 'null'
            WHEN DATE(START_MOMENT) < DATE "2010-01-01"               THEN 'pre_2010_junk'
            WHEN DATE(START_MOMENT) BETWEEN DATE "2024-01-01" AND DATE "2025-12-31" THEN 'in_window'
            ELSE 'other' END AS bucket,
       COUNT(*) AS rows
FROM {SES} GROUP BY bucket ORDER BY rows DESC''')

## 6 — Channel distribution (entry point)

**Why:** the channel is one of the entry-point drivers in the business question.
We need to know the share each channel has and spot any tiny channels that may
need filtering for stability.

In [ ]:
ch = q(f'''
SELECT CHANNEL, COUNT(*) AS n,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2) AS pct
FROM {SES} GROUP BY CHANNEL ORDER BY n DESC''')
ch

In [ ]:
fig,ax=plt.subplots(figsize=(11,3.6))
top = ch.head(10)
ax.barh(top.CHANNEL[::-1], top.n[::-1])
ax.set_title('Top channels by session volume'); plt.tight_layout(); plt.show()

## 7 — `service_type` (Mobile vs Fixed)

**Why:** business splits TOBi sessions into Mobile (M) and Fixed (F). Confirming
the mix matters for any segment-level reading.

In [ ]:
q(f'SELECT service_type, COUNT(*) AS n FROM {SES} GROUP BY service_type ORDER BY n DESC')

## 8 — Intent detection: `FIRST_INTENT` (top values) and `CONFIDENCE_LEVEL`

**Why:** these are the bot's read on each session. We want to:
- see the most common intents,
- understand the **distribution of `CONFIDENCE_LEVEL`** (it is stored as a string but is numeric).

In [ ]:
q(f'''SELECT FIRST_INTENT, COUNT(*) AS n FROM {SES}
GROUP BY FIRST_INTENT ORDER BY n DESC LIMIT 25''')

In [ ]:
# CONFIDENCE_LEVEL bucket: numeric string -> none / low / medium / high
q(f'''
WITH base AS (
  SELECT CASE
    WHEN CONFIDENCE_LEVEL IS NULL OR TRIM(CONFIDENCE_LEVEL)='' THEN 'unknown'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) IS NULL        THEN 'unknown'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) = 0            THEN 'none'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) < 0.5           THEN 'low'
    WHEN SAFE_CAST(CONFIDENCE_LEVEL AS FLOAT64) < 0.8           THEN 'medium'
    ELSE 'high' END AS confidence_band
  FROM {SES}
)
SELECT confidence_band, COUNT(*) AS n,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct
FROM base GROUP BY confidence_band ORDER BY n DESC''')

## 9 — `IS_FUNCTIONAL` (resolution / containment signal)

**Why:** this drives the FCR (first-contact resolution) metric. Confirm domain
values (`Yes`/`No`/blank/null).

In [ ]:
q(f'SELECT IS_FUNCTIONAL, COUNT(*) AS n FROM {SES} GROUP BY IS_FUNCTIONAL ORDER BY n DESC')

## 10 — Customer dimensions (`CUSTOMER_TYPE`, `SERVICE_STATUS`)

**Why:** segments matter to leadership (high-value vs Business). We confirm the
domain values present in the data so any segment cut is grounded.

In [ ]:
q(f'''SELECT CUSTOMER_TYPE, SERVICE_STATUS, COUNT(*) AS n
FROM {SES} GROUP BY CUSTOMER_TYPE, SERVICE_STATUS
ORDER BY n DESC LIMIT 20''')

## 11 — LOG token vocabulary

Each `LOG` row contains one token; the analysis reconstructs the trail by
concatenating tokens in `ROW_ID` order. The grammar is:
`S_`=Start (carries entity `E#` and intent `I#`), `R_`=Root, `M_`=Module,
`T_`=Tag, `E_`=End. `T_<digit><letter><roman>_<client>` is the routing/outcome.

**Why this matters for EDA:** the tag vocabulary determines how we classify
sessions. Below we count token types and show the top `T_` destinations.

In [ ]:
q(f'''
SELECT CASE
  WHEN STARTS_WITH(TRIM(LOG), 'S_') THEN 'S_state'
  WHEN STARTS_WITH(TRIM(LOG), 'R_') THEN 'R_root'
  WHEN STARTS_WITH(TRIM(LOG), 'M_') THEN 'M_module'
  WHEN STARTS_WITH(TRIM(LOG), 'T_') THEN 'T_tag'
  WHEN STARTS_WITH(TRIM(LOG), 'E_') THEN 'E_end'
  ELSE CONCAT('other:', SUBSTR(TRIM(LOG), 1, 2)) END AS token_type,
       COUNT(*) AS n
FROM {LOG} GROUP BY token_type ORDER BY n DESC''')

In [ ]:
# Top T_ routing destinations (the spine of the routing analysis)
q(f'''
SELECT TRIM(LOG) AS routing_target, COUNT(*) AS n_events,
       COUNT(DISTINCT SESSION_ID) AS n_sessions
FROM {LOG} WHERE STARTS_WITH(TRIM(LOG), 'T_')
GROUP BY routing_target ORDER BY n_sessions DESC LIMIT 20''')

In [ ]:
# Sample full reconstructed trails (eyeball the conversation flow)
q(f'''
SELECT SESSION_ID, COUNT(*) AS n_tokens,
       STRING_AGG(TRIM(LOG), ' > ' ORDER BY ROW_ID) AS flow_trail
FROM {LOG} GROUP BY SESSION_ID LIMIT 5''')

## 12 — Null rates on key fields

**Why:** any field used downstream needs its missingness understood, so we know
where the analysis must be defensive.

In [ ]:
q(f'''
SELECT
  COUNT(*) AS rows,
  ROUND(100*COUNTIF(FIRST_INTENT IS NULL OR FIRST_INTENT='')/COUNT(*),2) AS pct_first_intent_null,
  ROUND(100*COUNTIF(CONFIDENCE_LEVEL IS NULL OR CONFIDENCE_LEVEL='')/COUNT(*),2) AS pct_confidence_null,
  ROUND(100*COUNTIF(CHANNEL IS NULL OR CHANNEL='')/COUNT(*),2) AS pct_channel_null,
  ROUND(100*COUNTIF(IS_FUNCTIONAL IS NULL OR IS_FUNCTIONAL='')/COUNT(*),2) AS pct_is_functional_null,
  ROUND(100*COUNTIF(service_type IS NULL OR service_type='')/COUNT(*),2) AS pct_service_type_null,
  ROUND(100*COUNTIF(ANI IS NULL OR ANI='')/COUNT(*),2) AS pct_ani_null
FROM {SES}''')

## 13 — EDA findings & decisions

Findings (to be filled / confirmed when the cells above are run):

1. **Volumes:** sessions ~26M total; logs ~360M tokens with ~10x more rows than sessions.
2. **Date quality:** raw data contains `null` and `1900-01-01` rows. **Decision:** apply
   a clean analysis window (`DATE(START_MOMENT) BETWEEN '2024-01-01' AND '2025-12-31'`).
3. **Join integrity:** logs join cleanly to sessions; nearly all sessions have logs.
4. **`CONFIDENCE_LEVEL` is numeric stored as string**, dominated by `0`/blank.
   **Decision:** bucket into bands (none/low/medium/high) and avoid using it as a primary driver.
5. **`IS_FUNCTIONAL`** is `Yes`/`No`/blank/null. **Decision:** treat `Yes` as functional/contained.
6. **`service_type`** is `M`/`F` (Mobile/Fixed) with some null. Useful as a segment cut.
7. **`T_` vocabulary:** top destinations follow the pattern `T_<digit><letter><roman>_<client>`
   (matching the published mapping). **Decision:** decode tag outcome (1=Contained,
   2=Transferred), letter (A=Bot, B=Digital deflection, C=Assisted deflection, D=Abandon,
   E=Error, F=Service-change, 2A=Livechat, 2B=ACD), and Roman = support type
   (I=Non-tech, II=Tech, III=Commercial).
8. **Technical-topic detection from entity ids (`E#`)** — more accurate than text
   keyword matching. The exact entity list is documented in `docs/tag_mappings.md`.

These decisions are encoded in `standalone/session_master_query.sql`, which is the
single source of truth for the dashboard built on top of this EDA.

---
*Aligns with the data-science principle: validate first, analyse second. Open
items for the data-science lead to confirm: (1) is `general_difficulty` (intent
I8) appropriate as 'technical'? (2) treatment of Roman `IV/V/VI/VII` tags;
(3) Commercial (`III`) tag handling for technical topics; (4) the technical-
entity list itself (especially for Business sessions).*